# Session 7 – Customer Segmentation (Mall Customers Dataset)
Dataset: [Customer Segmentation Tutorial in Python – Kaggle](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python) (`Mall_Customers.csv`)

## 1. Load the Dataset and Display the First 10 Rows

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")

# Load the local copy (downloaded from Kaggle); fall back to a public mirror if it's missing
LOCAL = "Mall_Customers.csv"
URL = "https://raw.githubusercontent.com/tirthajyoti/Machine-Learning-with-Python/master/Datasets/Mall_Customers.csv"
df = pd.read_csv(LOCAL if os.path.exists(LOCAL) else URL)

print("Shape:", df.shape)
df.head(10)

In [ ]:
df.info()

## 2. Missing Values in Age, Annual Income and Spending Score

In [ ]:
cols = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
missing = df[cols].isnull().sum()

print("Missing values per column:")
for col, cnt in missing.items():
    print(f"  {col:<25}: {cnt}")
print(f"\nTotal missing values (all three columns): {missing.sum()}")

**Result:** There are **no missing values** in any of the three columns, so no imputation or row-dropping is needed.

## 3. Scatterplot – Annual Income vs Spending Score

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)",
                hue="Gender", palette={"Male": "#3F51B5", "Female": "#E91E63"},
                s=80, alpha=0.8, edgecolor="white")
plt.title("Customer Distribution: Annual Income vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend(title="Gender")
plt.show()

**Observation:** Customers are not spread uniformly – they form roughly **5 visible groups**: a dense centre (average income, average spending)
and four "corners" (low income/low spending, low income/high spending, high income/low spending, high income/high spending).
Gender does not appear to separate these groups.

## 4. Correlation Heatmap of Numerical Features
`CustomerID` is excluded – it is just a row identifier, not a real feature (and because the file is sorted by income, including it would show a misleading correlation).

In [ ]:
num_cols = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
corr = df[num_cols].corr()
print(corr.round(3))

plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, linewidths=1, cbar_kws={"label": "Pearson correlation"})
plt.title("Correlation Matrix – Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
# Find the strongest correlation (by absolute value), ignoring the diagonal
pairs = corr.where(~np.eye(len(corr), dtype=bool)).abs().unstack().dropna().sort_values(ascending=False)
f1, f2 = pairs.index[0]
print(f"Most strongly correlated pair: '{f1}' & '{f2}'  (r = {corr.loc[f1, f2]:.3f})")

### Interpretation
- The most strongly correlated pair is **Age and Spending Score (r ≈ -0.33)** – a **moderate negative** correlation: younger customers tend to have higher spending scores, older customers lower.
- **Annual Income vs Spending Score** is almost **zero (r ≈ 0.01)** – income alone does not predict spending. This is exactly why clustering is useful:
  the relationship is not linear, but the scatterplot shows distinct *groups*.
- **Age vs Annual Income** is also very weak (r ≈ -0.01).
- Overall, all correlations are weak-to-moderate, so the features carry largely independent information.

## 5. K-Means Clustering (k = 3) on Annual Income & Spending Score

In [ ]:
X = df[["Annual Income (k$)", "Spending Score (1-100)"]]

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X)

print("Customers per cluster:")
print(df["Cluster"].value_counts().sort_index())
df.head(10)

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x="Annual Income (k$)", y="Spending Score (1-100)",
                hue="Cluster", palette="Set2", s=80, edgecolor="white")
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=300, label="Centroids")
plt.title("Customer Segments (K-Means, k=3)")
plt.legend(title="Cluster")
plt.show()

profile = df.groupby("Cluster")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
profile["Customers"] = df["Cluster"].value_counts().sort_index()
profile

### Cluster Profiles
| Cluster | Customers | Avg Income (k$) | Avg Spending Score | Avg Age | Segment |
|---|---|---|---|---|---|
| **0** | 38 | 87.0 | 18.6 | 40.4 | **High income, low spenders** – "careful / savers"; target with premium-value or loyalty offers to unlock spending |
| **1** | 39 | 86.5 | 82.1 | 32.7 | **High income, high spenders** – "VIP / target" customers; younger, most valuable group for premium products |
| **2** | 123 | 44.2 | 49.8 | 40.3 | **Low-to-average income customers** – a broad mixed group; with only 3 clusters, K-Means merges the centre and the two low-income corners together |

**Note:** The scatterplot naturally shows ~5 groups; with k=3, the high-income customers split cleanly by spending, while all lower-income customers form one large cluster.
Using k=5 (e.g. picked via the Elbow method) would separate the low-income savers and low-income high spenders as well.